# Chapter 1, Exercise 2: WER by hand and WER/CER on real Arabic clips

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 1, Exercise 2.** Using Table 1.3, compute the WER by hand, reporting S, D, I, and N. Then recompute WER and CER with the companion notebook on its clips, stating the normalization applied and whether diacritics were scored.

**Note on data.** The book's companion clips were not included in the material supplied for these solutions. This notebook therefore uses openly licensed Arabic read speech from FLEURS (Arabic, `ar_eg` partition, test split, CC BY 4.0) as a stand-in, and it also lets you upload your own clips with a reference TSV. Numbers obtained on FLEURS are illustrative of the *method*; they are not the numbers the companion clips would give.

## Requirements

No GPU is required. A T4 GPU (free tier) makes transcription much faster and lets you use `openai/whisper-large-v3`. On CPU keep `N_CLIPS` at 20 or fewer.


## Part A: WER by hand from Table 1.3

Table 1.3 aligns the five-word reference with the recognizer output. We first count the operations by hand, then let `jiwer` confirm them.

In [1]:
!pip install -q jiwer python-Levenshtein

In [2]:
# Reference and hypothesis exactly as in Table 1.3 (transliterations in comments)
reference  = "وين أقرب محطة بنزين هنا"   # wēn aqrab maḥaṭṭat banzīn hunā
hypothesis = "أين أقرب محطة وقود"        # ayna aqrab maḥaṭṭat waqūd

# Hand count from Table 1.3:
#   position 1: وين -> أين      substitution (S)
#   position 2: أقرب = أقرب     correct
#   position 3: محطة = محطة     correct
#   position 4: بنزين -> وقود   substitution (S)
#   position 5: هنا -> (none)   deletion (D)
S, D, I = 2, 1, 0
N = len(reference.split())
print(f"S={S}, D={D}, I={I}, N={N}")
print(f"WER = (S+D+I)/N = ({S}+{D}+{I})/{N} = {(S+D+I)/N:.2f}  -> {100*(S+D+I)/N:.0f}%")

S=2, D=1, I=0, N=5
WER = (S+D+I)/N = (2+1+0)/5 = 0.60  -> 60%


In [3]:
import jiwer
out = jiwer.process_words(reference, hypothesis)
print("jiwer word-level:", "S=", out.substitutions, "D=", out.deletions, "I=", out.insertions,
      "N=", len(reference.split()), "WER=", round(out.wer, 3))
print(jiwer.visualize_alignment(out))

# Character Error Rate. Two conventions are common: counting spaces as characters (jiwer default)
# or ignoring them. Both are reported so that the convention is explicit.
cer_with_spaces = jiwer.process_characters(reference, hypothesis)
print("CER (spaces counted as characters): "
      f"S={cer_with_spaces.substitutions} D={cer_with_spaces.deletions} I={cer_with_spaces.insertions} "
      f"N={len(reference)} CER={cer_with_spaces.cer:.3f}")
import Levenshtein
r_ns, h_ns = reference.replace(' ', ''), hypothesis.replace(' ', '')
d = Levenshtein.distance(r_ns, h_ns)
print(f"CER (spaces ignored): edits={d} N={len(r_ns)} CER={d/len(r_ns):.3f}")

jiwer word-level: S= 2 D= 1 I= 0 N= 5 WER= 0.6
=== SENTENCE 1 ===

REF: وين أقرب محطة بنزين هنا
HYP: أين أقرب محطة  وقود ***
       S               S   D

=== SUMMARY ===
number of sentences: 1
substitutions=2 deletions=1 insertions=0 hits=2

mer=60.00%
wil=80.00%
wip=20.00%
wer=60.00%

CER (spaces counted as characters): S=5 D=5 I=0 N=23 CER=0.435
CER (spaces ignored): edits=9 N=19 CER=0.474


**Reading the result.** WER is 3/5 = 60 percent, exactly as the book states. The CER is lower (about 43 percent counting spaces, about 47 percent ignoring them) because وين and أين differ in a single letter, whereas بنزين and وقود share no letters. CER therefore distinguishes the near miss at position 1 from the total miss at position 4, which is why the book asks that both metrics be reported together.

## Part B: WER and CER on real clips

### B.1 Get some Arabic clips

Choose one of the two options below.

* **Option 1 (default):** download a small slice of the FLEURS Arabic test split from the Hugging Face Hub. No account is needed.
* **Option 2:** upload your own clips (WAV, 16 kHz preferred) plus a tab-separated file `refs.tsv` with two columns, `file` and `reference`.

In [4]:
!pip install -q "transformers>=4.40" torch soundfile huggingface_hub pyarrow pandas

In [5]:
import os, io, pandas as pd, numpy as np, soundfile as sf
N_CLIPS = int(os.environ.get("N_CLIPS", "20"))   # number of clips to score (keep small on CPU)

USE_UPLOAD = False   # set True to use your own clips (Option 2)
clips = []           # list of dicts: {"file": name, "audio": np.array, "sr": int, "reference": str}

if USE_UPLOAD:
    try:
        from google.colab import files
        uploaded = files.upload()   # upload the WAV files and refs.tsv together
    except ImportError:
        uploaded = {}
    refs = pd.read_csv("refs.tsv", sep="\t")
    for _, row in refs.iterrows():
        audio, sr = sf.read(row["file"])
        clips.append({"file": row["file"], "audio": audio, "sr": sr, "reference": row["reference"]})
else:
    from huggingface_hub import hf_hub_download
    # FLEURS ar_eg test split, parquet form (about 300 MB download, cached by the Hub client)
    path = hf_hub_download("google/fleurs", "parquet-data/ar_eg/test-00000-of-00001.parquet",
                           repo_type="dataset")
    import pyarrow.parquet as pq
    table = pq.ParquetFile(path)
    batch = next(table.iter_batches(batch_size=N_CLIPS))
    df = batch.to_pandas()
    for _, row in df.iterrows():
        a = row["audio"]
        audio, sr = sf.read(io.BytesIO(a["bytes"]))
        clips.append({"file": a.get("path") or row.get("path", ""), "audio": audio, "sr": sr,
                      "reference": row["raw_transcription"]})
print(f"{len(clips)} clips loaded; sample rate of first clip: {clips[0]['sr']} Hz")
print("First reference:", clips[0]["reference"])

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


20 clips loaded; sample rate of first clip: 16000 Hz
First reference: تشكلت في المحيط الأطلسي اليوم عاشر عاصفة مُسماة لموسم الأعاصير الأطلسية، العاصفة شبه الاستوائية جيري.


### B.2 Transcribe with Whisper

`openai/whisper-small` (244 M parameters) is used by default. On the free CPU runtime it takes roughly 10 to 30 seconds per clip; with a T4 GPU (Runtime > Change runtime type > T4 GPU) it is a few seconds per clip and you can switch to `openai/whisper-large-v3` for a stronger baseline.

In [6]:
import torch, warnings, transformers
warnings.filterwarnings("ignore"); transformers.logging.set_verbosity_error()
from transformers import pipeline
MODEL = os.environ.get("ASR_MODEL", "openai/whisper-small")
device = 0 if torch.cuda.is_available() else -1
asr = pipeline("automatic-speech-recognition", model=MODEL, device=device,
               generate_kwargs={"language": "arabic", "task": "transcribe"})

hyps = []
for c in clips:
    audio = c["audio"] if c["audio"].ndim == 1 else c["audio"].mean(axis=1)
    if c["sr"] != 16000:
        import librosa
        audio = librosa.resample(audio.astype(np.float32), orig_sr=c["sr"], target_sr=16000)
    result = asr({"raw": audio.astype(np.float32), "sampling_rate": 16000})
    hyps.append(result["text"].strip())
for c, h in list(zip(clips, hyps))[:3]:
    print("REF:", c["reference"]); print("HYP:", h); print()

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:  80%|████████  | 385/479 [00:00<00:00, 3815.96it/s]

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 4237.95it/s]

REF: تشكلت في المحيط الأطلسي اليوم عاشر عاصفة مُسماة لموسم الأعاصير الأطلسية، العاصفة شبه الاستوائية جيري.
HYP: تشكلت في المحيط الأطلسي اليوم عشر عصفة مسمى لموسى من الأعصير الأطلسية العصفة شبه الاستوائية جري

REF: تغادر الحافلات المحطة الداخلية بين المناطق (عبر النهر) في خلال اليوم على الرغم من أن معظمها وخاصة المتجهة منها إلى الشرق وجاكار/بومثانج تغادر بين 06:30 و 07:30.
HYP: تغادر حافلات المحطة دخلية بين المناطق عبر النهر في خلال اليوم على الرغم من أن معظمها وخاصة المتجه منها إلى الشرق وجكارا بومثانج تغادر بينها 36 و 37

REF: يتمُّ دعْم التعلُّم التفاعليّ في البرنامج داخليًا ويهدف إلى طرح الأسئلة والتحفيز وشرح الإجراءاتِ التي قد يكون من الصعب على الطالب التعامل معها بمفرده.
HYP: يتم دعم التعلم التفاعلية في البرنامج داخليا ويهدف إلى طرح الأسئلة والتحفيظ وشرح الإجراءات التي قد يكون من الصعب على الطالب التعامل معها بمفرده



### B.3 State the normalization, then score

Two normalization policies are applied to **both** reference and hypothesis, and both results are reported. This follows the Reproducibility Note of Chapter 1 and Table 4.2.

| Policy | Rules applied | Diacritics scored? |
|---|---|---|
| `strict` | Unicode NFC; remove punctuation; collapse whitespace; remove tatweel; **remove diacritics** (fatḥa, kasra, ḍamma, sukūn, shadda, tanwīn) | No |
| `lenient` | everything in `strict`, plus: unify alif forms (أ إ آ ٱ to ا); tāʾ marbūṭa ة to ه; alif maqṣūra ى to ي; Arabic-Indic digits to Western digits | No |

Diacritics are **not** scored under either policy, because neither FLEURS transcripts nor Whisper output are consistently diacritized. The `lenient` policy merges forms that are not linguistically identical (see Section 4.2), so it should be reported as a scoring convention, not as a claim of equivalence.

In [7]:
import re, unicodedata
DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
TATWEEL = "\u0640"
PUNCT = re.compile(r"[\u060C\u061B\u061F\u066A-\u066D\u06D4!-/:-@\[-`{-~\u201C\u201D\u2018\u2019\u00AB\u00BB]")
AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def normalize(text, policy="strict"):
    t = unicodedata.normalize("NFC", text)
    t = t.replace(TATWEEL, "")
    t = DIACRITICS.sub("", t)
    t = PUNCT.sub(" ", t)
    if policy == "lenient":
        t = re.sub("[أإآٱ]", "ا", t)
        t = t.replace("ة", "ه").replace("ى", "ي")
        t = t.translate(AR_DIGITS)
    t = re.sub(r"\s+", " ", t).strip()
    return t

refs = [c["reference"] for c in clips]
for policy in ["strict", "lenient"]:
    R = [normalize(r, policy) for r in refs]
    H = [normalize(h, policy) for h in hyps]
    w = jiwer.process_words(R, H)
    ch = jiwer.process_characters(R, H)
    N = sum(len(r.split()) for r in R)
    print(f"[{policy}] model={MODEL} clips={len(R)}  "
          f"WER={100*w.wer:.2f}%  (S={w.substitutions} D={w.deletions} I={w.insertions} N={N})  "
          f"CER={100*ch.cer:.2f}%")

[strict] model=openai/whisper-small clips=20  WER=22.14%  (S=84 D=5 I=2 N=411)  CER=6.05%
[lenient] model=openai/whisper-small clips=20  WER=21.65%  (S=82 D=5 I=2 N=411)  CER=5.92%


In [8]:
# Per-clip table (strict policy) so that outliers can be inspected
rows = []
for c, h in zip(clips, hyps):
    r_n, h_n = normalize(c["reference"]), normalize(h)
    o = jiwer.process_words(r_n, h_n)
    rows.append({"file": os.path.basename(str(c["file"])), "N": len(r_n.split()),
                 "S": o.substitutions, "D": o.deletions, "I": o.insertions, "WER%": round(100*o.wer, 1)})
pd.DataFrame(rows).sort_values("WER%", ascending=False).head(10)

,file,N,S,D,I,WER%
8,10361044285640817256.wav,13,6,1,0,53.8
0,10019390257075855641.wav,15,7,0,1,53.3
18,10726395627256131145.wav,26,11,0,0,42.3
6,10340788584398631536.wav,17,6,0,1,41.2
9,10366294495925707527.wav,22,8,0,0,36.4
10,10372562911047932559.wav,27,8,1,0,33.3
1,10035416633083120372.wav,30,7,2,0,30.0
11,10394942145902478734.wav,21,5,0,0,23.8
13,10459237854135592487.wav,10,2,0,0,20.0
3,1021284281927612037.wav,18,2,1,0,16.7


### B.4 How to report the result

A complete statement looks like this (fill in your own numbers from the cells above):

> Whisper-small (zero-shot, `language=ar`, greedy decoding, `transformers` pipeline) on the first *N* clips of the FLEURS `ar_eg` test split gave WER = __ % and CER = __ % under the `strict` policy (punctuation and diacritics removed, no letter unification) and WER = __ % / CER = __ % under the `lenient` policy (additionally unifying alif, tāʾ marbūṭa, alif maqṣūra, and digits). Diacritics were not scored. Both reference and hypothesis were normalized with the same script (this notebook).

The gap between the two policies is itself informative: it is the share of "errors" that are spelling conventions rather than misrecognitions.